# 1. What is Data Validation?

### Concept & Definition
Data validation is the process of ensuring that data adheres to defined quality, domain, and structural criteria before it is ingested into machine learning pipelines or analytical engines.

### Real-World / Business Example
In a fintech onboarding system, an applicant entering an age of `-25` or a loan application date from the year `1890` must be flagged immediately to prevent invalid processing.

### ML Impact
Validating data prevents corrupted inputs from degrading model performance, causing silent failures, or producing erroneous predictions in production.

In [1]:
import numpy as np
import pandas as pd

# Load dataset
df = pd.read_csv("Customer_Data.csv")

print("=== Dataset Initial Inspection ===")
print(df.info())
display(df.head(5))

=== Dataset Initial Inspection ===
<class 'pandas.DataFrame'>
RangeIndex: 1010 entries, 0 to 1009
Data columns (total 9 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   CustomerID      1010 non-null   str    
 1   Age             956 non-null    float64
 2   Gender          962 non-null    str    
 3   TenureYears     1010 non-null   int64  
 4   MonthlyCharges  967 non-null    str    
 5   TotalCharges    1010 non-null   float64
 6   ContractType    1010 non-null   str    
 7   PaymentMethod   1010 non-null   str    
 8   Churn           1010 non-null   str    
dtypes: float64(2), int64(1), str(6)
memory usage: 114.3 KB
None


,CustomerID,Age,Gender,TenureYears,MonthlyCharges,TotalCharges,ContractType,PaymentMethod,Churn
0,CUST-1000,34.0,Male,3,$29.85,1098.216202,Two year,Credit card,No
1,CUST-1001,150.0,Female,10,$56.95,4544.186090,month to month,Mailed check,Yes
2,CUST-1002,52.0,M,2,$105.50,6144.766598,One year,Credit card,No
3,CUST-1003,45.0,Female,10,$56.95,7529.311738,Month-to-month,Bank transfer,Yes
4,CUST-1004,25.0,M,3,$29.85,225.003059,One year,Mailed check,Yes


# 2. Range Validation

### Concept & Definition
Verifies that numerical attributes fall strictly within acceptable upper and lower bound limits (e.g., $Age \ge 0$, $Salary \ge 0$, $0 \le Percentage \le 100$).

### Real-World / Business Example
A retail churn dataset where `Age` cannot be negative or over 120, and `MonthlyCharges` cannot be negative.

In [2]:
# Range Validation Rules
df["Age"] = pd.to_numeric(df["Age"], errors="coerce")
df["MonthlyCharges"] = pd.to_numeric(
    df["MonthlyCharges"].astype(str).str.replace("$", ""), errors="coerce"
)

# Identify range violations
invalid_age = df[(df["Age"] < 0) | (df["Age"] > 120)]
invalid_charges = df[df["MonthlyCharges"] < 0]

print(f"Invalid Age Records Count: {len(invalid_age)}")
print(f"Invalid MonthlyCharges Records Count: {len(invalid_charges)}")

Invalid Age Records Count: 47
Invalid MonthlyCharges Records Count: 0


# 3. Type Validation

### Concept & Definition
Ensures that each feature column contains values matching its expected data type (e.g., integers, floats, booleans, datetime).

### Real-World / Business Example
Detecting string inputs like `"Thirty"` in a numerical `Age` field.

In [3]:
# Type Validation Function
def validate_column_types(dataframe, expected_types):
    type_report = {}
    for col, expected_type in expected_types.items():
        is_valid = dataframe[col].apply(lambda x: isinstance(x, expected_type) if pd.notnull(x) else True).all()
        type_report[col] = "Valid" if is_valid else "Invalid Type Detected"
    return type_report

expected = {"CustomerID": str, "TenureYears": (int, float, np.integer, np.floating)}
print("=== Type Validation Report ===")
print(validate_column_types(df, expected))

=== Type Validation Report ===
{'CustomerID': 'Valid', 'TenureYears': 'Valid'}


# 4. Format Validation

### Concept & Definition
Checks whether text strings conform to mandatory regular expression patterns (e.g., Email formats, Phone Numbers, Zip Codes, Customer ID structures).

In [4]:
# Validate CustomerID format (e.g., must match pattern 'CUST-XXXX')
import re

pattern = r"^CUST-\d{4}$"
invalid_ids = df[~df["CustomerID"].astype(str).str.match(pattern, na=False)]

print(f"Non-matching CustomerID Formats Count: {len(invalid_ids)}")
if len(invalid_ids) > 0:
    display(invalid_ids[["CustomerID"]].head(5))

Non-matching CustomerID Formats Count: 0


# 5. Category Validation

### Concept & Definition
Verifies that categorical variables contain only values from a predefined set of allowed domain categories (e.g., `Gender` $\in$ `{"Male", "Female", "Other"}`).

In [5]:
# Define permitted categories
allowed_genders = {"Male", "Female", "Other"}
allowed_contracts = {"Month-to-Month", "One Year", "Two Year"}

invalid_gender = df[~df["Gender"].isin(allowed_genders) & df["Gender"].notnull()]
invalid_contract = df[~df["ContractType"].isin(allowed_contracts) & df["ContractType"].notnull()]

print(f"Invalid Gender Entries: {len(invalid_gender)}")
print(f"Invalid Contract Entries: {len(invalid_contract)}")

Invalid Gender Entries: 159
Invalid Contract Entries: 1010


# 6. Null Validation

### Concept & Definition
Assesses whether key mandatory features (such as primary keys or critical targets) contain unpermitted null or empty values.

In [6]:
# Ensure mandatory primary identifiers and targets are never null
mandatory_cols = ["CustomerID", "Churn"]
null_violations = df[mandatory_cols].isnull().sum()

print("=== Mandatory Column Null Violations ===")
print(null_violations)

=== Mandatory Column Null Violations ===
CustomerID    0
Churn         0
dtype: int64


# 7. Business Rule Validation

### Concept & Definition
Evaluates multi-column logical dependencies defined by domain requirements (e.g., `TotalCharges` $\approx$ `MonthlyCharges` $\times$ `TenureYears` $\times 12$).

In [7]:
# Business rule check: TotalCharges should not be lower than single MonthlyCharges if tenure > 0
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

rule_violations = df[(df["TenureYears"] > 0) & (df["TotalCharges"] < df["MonthlyCharges"])]
print(f"Business Rule Violations (TotalCharges < MonthlyCharges when tenure > 0): {len(rule_violations)}")

Business Rule Violations (TotalCharges < MonthlyCharges when tenure > 0): 0


# 8. Referential Validation

### Concept & Definition
Checks foreign key relationships between tables to ensure all reference keys exist in parent tables.

In [8]:
# Simulating referential integrity check between primary customer table and external active ID database
valid_active_ids = set(df["CustomerID"].dropna().unique()[:800]) # Master active set

unlinked_records = df[~df["CustomerID"].isin(valid_active_ids)]
print(f"Referential Integrity Check - Unlinked Customers Count: {len(unlinked_records)}")

Referential Integrity Check - Unlinked Customers Count: 200


# 9. Date Validation

### Concept & Definition
Ensures temporal logical consistency (e.g., `Date of Joining` cannot be before `Date of Birth`, and dates cannot be set in the future).

In [9]:
# Date validation example using simulated birth and joining dates
df_dates = pd.DataFrame({
    "DOB": pd.to_datetime(["1990-05-15", "2005-01-20", "1985-11-30"]),
    "JoiningDate": pd.to_datetime(["2015-06-01", "2000-01-01", "2010-08-15"]) # Row 2 invalid
})

invalid_temporal_records = df_dates[df_dates["JoiningDate"] < df_dates["DOB"]]
print("=== Temporal Violation Records (Joining Before Birth) ===")
display(invalid_temporal_records)

=== Temporal Violation Records (Joining Before Birth) ===


,DOB,JoiningDate
1,2005-01-20,2000-01-01


# 10. Constraint Validation

### Concept & Definition
Enforces structural uniqueness and integrity constraints (e.g., unique primary key constraints across the dataset).

In [10]:
# Constraint validation: CustomerID must be 100% unique across rows
is_unique = df["CustomerID"].is_unique
duplicate_count = df["CustomerID"].duplicated().sum()

print(f"Primary Key Uniqueness Constraint Met: {is_unique}")
print(f"Duplicate CustomerIDs Found: {duplicate_count}")

# Checkpoint Export
df.to_csv("Cleaned_Validated_Data.csv", index=False)
print("Notebook 05 execution completed successfully!")

Primary Key Uniqueness Constraint Met: False
Duplicate CustomerIDs Found: 10
Notebook 05 execution completed successfully!
